## Расчет и анализ метрик для благотворительного фонда АиФ. Доброе сердце

У фонда есть благотворители, которые поддерживают и помогают организации. Фонду важно знать о них больше, чтобы эффективнее с ними работать и коммуницировать.

**Задача исследования** — провести когортный анализ и рассчитать Retention Rate, LTV, средний чек, а также MAU.

Ссылка на дашборд с рассчитанными метриками: https://datalens.yandex/fcoq136cc1l8y?_share_link=public

На дашборде представлена визуализация определенных метрик и аналитические выводы:

- MAU (количество активных благотворителей за месяц)
- Retention Rate в зависимости от пола благотворителя
- Средний чек и кол-во пожертвований в зависимости от пола благотворителя
- LTV и кол-во благотворителей в крупных городах
- Детализация среднего чек и кол-ва пожертвований в зависимости от пола благотворителя по месяцам

Используются данные за весь период с 01.01.2021 по 01.03.2024

### SQL-код расчета метрик для благотворительного фонда

#### MAU (количество активных благотворителей за месяц)

```
WITH active_users AS
(SELECT 
    DATE_TRUNC('month', "OrderFirstActionDateTimeUtc")::date as "Месяц",
    "OrderCustomerIdsMindboxId" as user_id,
    COUNT(DISTINCT "OrderIdsMindboxId") as orders
FROM aif_data.order
WHERE "OrderLineStatusIdsExternalId" = 'Paid'
GROUP BY 1, 2
HAVING COUNT(DISTINCT "OrderIdsMindboxId") > 0)

SELECT
    "Месяц",
    COUNT(DISTINCT user_id) as MAU
FROM active_users
GROUP BY 1
ORDER BY 1
```

#### Retention Rate по когортам пола благотворителя

```
WITH active_users_august as 
(SELECT
    DATE_TRUNC('month', "OrderFirstActionDateTimeUtc")::date as "Август",
    "OrderCustomerIdsMindboxId" as user_id,
    "CustomerSex" as gender
FROM aif_data.order ao
LEFT JOIN aif_data.id_donor aid ON ao."OrderCustomerIdsMindboxId" = aid."CustomerIdsMindboxId"
WHERE "OrderLineStatusIdsExternalId" = 'Paid' AND DATE_TRUNC('month', "OrderFirstActionDateTimeUtc")::date = '01.08.2022' AND "CustomerSex" IS NOT NULL),

active_users_after_august as
(SELECT
    DATE_TRUNC('month', "OrderFirstActionDateTimeUtc")::date as "Месяц пожертвования",
    "OrderCustomerIdsMindboxId" as user_id,
    "CustomerSex" as gender
FROM aif_data.order ao
LEFT JOIN aif_data.id_donor aid ON ao."OrderCustomerIdsMindboxId" = aid."CustomerIdsMindboxId"
WHERE "OrderLineStatusIdsExternalId" = 'Paid' AND DATE_TRUNC('month', "OrderFirstActionDateTimeUtc")::date >= '01.08.2022' AND "CustomerSex" IS NOT NULL
ORDER BY "Месяц пожертвования"),

month_retention AS 
(SELECT
    aua."Август",
    aua.user_id,
    aua.gender,
    aa."Месяц пожертвования",
    ("Месяц пожертвования" - "Август")/30 as montly_since_install
FROM active_users_august aua
JOIN active_users_after_august aa ON aua.user_id = aa.user_id)

SELECT
    gender as "Пол благотворителя",
    montly_since_install as "Месяцев после медийной кампании",
    COUNT(DISTINCT user_id) as retained_users,
    COUNT(DISTINCT user_id)::numeric / MAX(COUNT(DISTINCT user_id)) OVER (PARTITION BY gender ORDER BY montly_since_install) as retention_rate
FROM month_retention 
GROUP BY 1, 2
ORDER BY 1, 2
```

#### Средний чек и кол-во заказов благотворителей мужчин

```
SELECT
    DATE_TRUNC('month', "OrderFirstActionDateTimeUtc")::date as "Месяц",
    "CustomerSex" as gender,
    COUNT(DISTINCT "OrderIdsMindboxId") as "Кол-во заказов",
    SUM("OrderTotalPrice"::numeric) / COUNT(DISTINCT "OrderIdsMindboxId") as "Средний чек"
FROM aif_data.order ao
LEFT JOIN aif_data.id_donor aid ON ao."OrderCustomerIdsMindboxId" = aid."CustomerIdsMindboxId"
WHERE "OrderLineStatusIdsExternalId" = 'Paid' AND "CustomerSex" = 'male'
GROUP BY 1, 2
ORDER BY 1, 2
```

#### Средний чек и кол-во заказов благотворителей женщин

```
SELECT
    DATE_TRUNC('month', "OrderFirstActionDateTimeUtc")::date as "Месяц",
    "CustomerSex" as gender,
    COUNT(DISTINCT "OrderIdsMindboxId") as "Кол-во заказов",
    SUM("OrderTotalPrice"::numeric) / COUNT(DISTINCT "OrderIdsMindboxId") as "Средний чек"
FROM aif_data.order ao
LEFT JOIN aif_data.id_donor aid ON ao."OrderCustomerIdsMindboxId" = aid."CustomerIdsMindboxId"
WHERE "OrderLineStatusIdsExternalId" = 'Paid' AND "CustomerSex" = 'female'
GROUP BY 1, 2
ORDER BY 1, 2
```

#### Средний чек благотворителей по когортам в зависимости от пола

```
SELECT
    "CustomerSex" as "Пол благотворителя",
    COUNT(DISTINCT "OrderIdsMindboxId") as "Кол-во заказов",
    SUM("OrderTotalPrice"::numeric) / COUNT(DISTINCT "OrderIdsMindboxId") as "Средний чек"
FROM aif_data.order ao
LEFT JOIN aif_data.id_donor aid ON ao."OrderCustomerIdsMindboxId" = aid."CustomerIdsMindboxId"
WHERE "OrderLineStatusIdsExternalId" = 'Paid' AND "CustomerSex" IS NOT NULL
GROUP BY 1
ORDER BY 1
```

#### LTV по городам благотворителя

```
SELECT
    "CustomerAreaName" as "Город благотворителя",
    COUNT(DISTINCT "OrderCustomerIdsMindboxId") as "Кол-во благотворителей",
    SUM("OrderTotalPrice"::numeric) / COUNT(DISTINCT "OrderIdsMindboxId") as "LTV"
FROM aif_data.order ao
LEFT JOIN aif_data.id_donor aid ON ao."OrderCustomerIdsMindboxId" = aid."CustomerIdsMindboxId"
WHERE "OrderLineStatusIdsExternalId" = 'Paid' AND "CustomerAreaName" IN ('Москва', 'Санкт-Петербург', 'Екатеринбург')
GROUP BY 1
ORDER BY 1
```